# Метрики відстані та матриці відстаней (Breast Cancer)

Імпорти

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

### 1. Завантажуємо дані

In [ ]:
data = load_breast_cancer()
print(data.DESCR[:1000])

### 2. Робимо DataFrame

In [ ]:
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df.head()

### 3. Інформація про дані

In [ ]:
df.info()

Всі стовпці числові (float), пропущених значень немає - всюди по 569 значень.

### 4. Описові статистики

In [ ]:
df.describe()

Видно що ознаки в дуже різних масштабах (наприклад area в сотнях/тисячах, а smoothness в сотих), тому потрібна стандартизація.

### 5. Стандартизація

In [ ]:
X = df.drop('target', axis=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.describe().round(2).iloc[:3]

Тепер середнє у всіх ознак близько 0, а стандартне відхилення 1.

### 6. Точкові діаграми

Ознак дуже багато (30), тому pairplot по всіх будувати нема сенсу. Візьму кілька перших ознак.

In [ ]:
cols = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'target']
sns.pairplot(df[cols], hue='target')
plt.show()

Видно що ознаки розміру (radius, perimeter, area) сильно зв'язані між собою, і по них два класи (хворі/здорові) досить добре розділяються.

### 7. Матриці відстаней

Рахуємо матриці відстаней між об'єктами для різних метрик. Беру не всі 569 рядків, а перші 30, щоб матриці було видно на графіку.

In [ ]:
sample = X_scaled.values[:30]

metrics = ['cityblock', 'cosine', 'euclidean', 'l1', 'manhattan']
distance_matrices = {}
for m in metrics:
    distance_matrices[m] = pairwise_distances(sample, metric=m)

distance_matrices['euclidean'].shape

### 8. Візуалізація матриць (heatmap)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, m in enumerate(metrics):
    sns.heatmap(distance_matrices[m], ax=axes[i], cmap='viridis')
    axes[i].set_title(m)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

Помітно що cityblock, l1 і manhattan дають однакові матриці - це по суті одна і та сама метрика (сума модулів різниць). euclidean схожа на них але значення трохи інші. А cosine відрізняється найбільше, бо вона міряє не відстань а кут між векторами.

### 9. Висновок

В цій роботі я рахувала матриці відстаней для набору Breast Cancer різними метриками.

Спочатку завантажила дані, зробила DataFrame, подивилась info() і describe(). Виявилось що ознаки в дуже різних масштабах, тому я зробила стандартизацію - без неї метрики відстані працювали б неправильно, бо великі ознаки (area) перебивали б маленькі.

Потім порахувала матриці відстаней для метрик cityblock, cosine, euclidean, l1 і manhattan та вивела їх через heatmap.

По результатах видно що cityblock, l1 і manhattan - це фактично одна метрика і матриці у них однакові. euclidean дуже схожа на них. А cosine сильно відрізняється від інших, бо вона враховує не відстань між точками, а напрямок векторів (кут).

Тобто для цих даних евклідова і манхеттенська метрики дають схожий результат, а косинусна - інший погляд на схожість об'єктів. Який вибір метрики краще - залежить від задачі.